# Exercise 3 — Connecting Python and SQL: An End-to-End ETL

**Time:** ~25 minutes

**Goal:** Build a full **ETL pipeline** in Python:

> `[ SQLite ] → [ Extract via SQL ] → [ Transform in pandas ] → [ Load to CSV ]`

This is the same ETL diagram from Session 1 — only now **you** write the pipeline code instead of clicking around in a BI tool.

**Prereq:** Run `python init_orders.py` once to create `cafe_data.db`.

## Why connect Python to SQL?

- SQL is **best at filtering and joining** at the database — close to the data, fast.
- pandas is **best at flexible transformations** — pivots, complex KPIs, what-ifs.
- Real teams use **both**: SQL extracts a focused slice, pandas reshapes it.
- Python lets you **automate** the pipeline (run nightly, send a report, refresh a dashboard).

Schema in `cafe_data.db`:

```
menu      (id, name, price, category)
customers (id, name, signup_date)
orders    (id, customer_id, menu_id, quantity, order_date)
```

## EXTRACT — pull joined data out of SQLite

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('cafe_data.db')

# Sanity check — what tables are in there?
pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn,
)

**TODO:** Write a SQL query that **joins** `orders` with `menu` and `customers`, returning one row per order line with these columns:

`order_id, order_date, customer_name, product, category, quantity, price, revenue`

Note `revenue = quantity * price` — compute it directly in SQL.

*Recall from last session:* `INNER JOIN <table> <alias> ON ...`

In [ ]:
query = """
    -- YOUR SQL HERE
    SELECT 1
"""

df = pd.read_sql_query(query, conn)
df.head()
# Expected: 95 rows, 8 columns

In [ ]:
print(f"Extracted {len(df)} order lines")
df.info()

## TRANSFORM — compute three KPIs in pandas

Recall: KPIs are **measurable, actionable, aligned with goals**. Café Cozy's owner wants to know:

1. **Top-selling product by revenue** — what should we never run out of?
2. **Revenue by category** — is the bakery worth the kitchen space?
3. **Average basket size per order** — are people upselling, or just grabbing one drink?

### KPI 1 — Top-selling product by revenue

**TODO:** Group by `product`, sum `revenue`, sort descending.

*Hint:* `df.groupby('col')['other_col'].sum().sort_values(ascending=False)`

In [ ]:
# YOUR CODE HERE
top_products = None
top_products

### KPI 2 — Revenue and order count by category

**TODO:** Group by `category`. Use `.agg()` to compute **two** aggregations at once: total revenue and number of order lines.

*Hint:*
```python
df.groupby('category').agg(
    revenue=('revenue', 'sum'),
    order_lines=('order_id', 'count'),
)
```

In [ ]:
# YOUR CODE HERE
by_category = None
by_category

### KPI 3 — Average basket size per order

An "order" is one `order_id`. Basket size = total revenue for that order.

**TODO:**
1. Group by `order_id`, sum `revenue` per order → `basket_per_order`
2. Take `.mean()` and `.median()` of those baskets

In [ ]:
# YOUR CODE HERE
basket_per_order = None
avg_basket    = None
median_basket = None

print(f"Average basket size: €{avg_basket}")
print(f"Median basket size:  €{median_basket}")
print(f"Total orders:        {len(basket_per_order) if basket_per_order is not None else '?'}")

## LOAD — save results so somebody else can use them

A KPI nobody can see is a KPI that does nothing. We'll write our results to CSV — that's the file format dashboards (Power BI, Looker, Data Studio) and spreadsheets all accept.

In [ ]:
# TODO: save each KPI to its own CSV
# Useful: df.to_csv('filename.csv', index=False) — or for a Series, df.to_csv('filename.csv', header=['column_name'])

# top_products.to_csv(...)
# by_category.to_csv(...)

# Bonus: build a one-line summary DataFrame with avg_basket, median_basket, total_orders, total_revenue

print('Done.')

In [ ]:
# Always close DB connections when done
conn.close()

## 🚀 Stretch goals — pick one

1. **Most loyal customer** — group by `customer_name`, count distinct `order_id`s, find the customer with the most orders.
2. **Daily revenue trend** — group by `order_date`, sum revenue, then `.plot()` it.
3. **Filter then aggregate** — what's the top-selling Bakery item? Use `df[df['category'] == 'Bakery']` first.
4. **Write back to the DB** — save your KPI table to a new SQLite table: `df.to_sql('kpi_top_products', conn, if_exists='replace')`.

In [ ]:
# YOUR STRETCH CODE HERE


### Where this fits — Data Science Lifecycle recap

Today's pipeline mapped to last week's lifecycle slide:

| Phase | What we did |
|---|---|
| Business understanding | "Owner wants top seller, category revenue, basket size" |
| Data mining / extract | SQL JOIN across orders + menu + customers |
| Data cleaning | Already clean here — but Exercise 2 showed how |
| Data exploration | `.head()`, `.info()`, `.describe()` |
| Feature engineering | `revenue = quantity * price` |
| Predictive modeling | *(skipped — not every analysis needs ML)* |
| Data visualization | The CSVs feed a dashboard — Power BI / Looker takes it from here |

**You just built a real data pipeline.** Every analytics team in industry runs scripts like this — often automated to run nightly. The shape doesn't change.